# Lab 7: Draw the reviews
Same Airbnb table as in class, but other columns: mostly `number_of_reviews` (Part 1), and in Part 2 also the rating.

If the libraries are missing, run `pip install matplotlib seaborn` first.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# change the list if your file has other date columns
listings = pd.read_csv('data/listings_clean.csv',
                       parse_dates=['last_scraped', 'first_review', 'last_review'])
if 'has_reviews' not in listings:                       # in case my file has no such column
    listings['has_reviews'] = listings['number_of_reviews'] > 0
reviews = listings['number_of_reviews']
print(listings.shape)

# Part 1: Practise

## Task 1: Set up, and the first histogram

In [ ]:
# Pure matplotlib: just the bars. No title and no axis names, so I have to guess what I am looking at.
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(reviews, bins=50)
plt.show()
# I see one very tall bar on the left (about 3300 listings with 0 to 26 reviews). The bars get lower and
# lower, and a thin tail goes to the right, all the way to 1293 reviews.

In [ ]:
# Seaborn: same shape, cleaner look from the whitegrid theme, plus the text I add myself.
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=listings, x='number_of_reviews', bins=50, ax=ax)
ax.set_title('Most listings have few reviews, a few have more than 1000')
ax.set_xlabel('Number of reviews')
ax.set_ylabel('Number of listings')
plt.tight_layout()
plt.show()
# The difference: seaborn gives the same picture with a cleaner style, and now the chart says what
# it shows (title) and what the axes mean.

## Task 2: Mean, median and the fence

In [ ]:
median = reviews.median()
mean = reviews.mean()
q1, q3 = reviews.quantile(0.25), reviews.quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print(median, round(mean, 1), upper)        # 9.0 35.8 69.5

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=listings, x='number_of_reviews', bins=50, ax=ax)
ax.axvline(median, color='green', linestyle='--', label=f'Median ({median:.0f})')
ax.axvline(mean, color='orange', linestyle='--', label=f'Mean ({mean:.1f})')
ax.axvline(upper, color='red', linestyle='--', label=f'Upper fence ({upper:.1f})')
ax.legend()
ax.set_title('The mean is pulled to the right of the median by a few popular listings')
ax.set_xlabel('Number of reviews')
ax.set_ylabel('Number of listings')
plt.tight_layout()
plt.show()
# The upper fence (69.5) is furthest to the right. The mean (35.8) is to the right of the median (9),
# because the few listings with very many reviews pull it up. The median is not pulled.

## Task 3: With and without the outliers

In [ ]:
listings['reviews_outlier'] = reviews > upper
n_out = listings['reviews_outlier'].sum()
print(n_out)                                 # 582

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=listings, x='number_of_reviews', bins=50, ax=axes[0])
axes[0].set_title('All 4574 listings: a long tail up to 1293')
sns.histplot(data=listings[~listings['reviews_outlier']], x='number_of_reviews', bins=50, ax=axes[1])
axes[1].set_title(f'Without the {n_out} outliers: nothing above 69')
for ax in axes:
    ax.set_xlabel('Number of reviews')
    ax.set_ylabel('Number of listings')
plt.tight_layout()
plt.show()
# The left chart is stretched by the long tail, so the shape of most listings is squeezed into the first bars.
# The right chart has a much shorter x-axis, so I can see the shape of the other 3992 listings:
# most are still on the left, but there is a visible slope. The tallest bar is still the first one.

## Task 4: The boxplot

In [ ]:
big = listings[listings['room_type'].isin(['Entire home/apt', 'Private room'])]
outside = (big['number_of_reviews'] > 300).sum()
print(len(big), outside)                                    # 4547 listings, 90 outside the picture
print(big.groupby('room_type')['number_of_reviews'].median())   # Entire home/apt 8, Private room 12

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=big, x='room_type', y='number_of_reviews', order=['Entire home/apt', 'Private room'], ax=ax)
ax.set_ylim(0, 300)
ax.set_title(f'Private rooms have a higher median ({12}) than entire homes ({8})\n({outside} listings with more than 300 reviews are outside the picture)')
ax.set_xlabel('Room type')
ax.set_ylabel('Number of reviews')
plt.tight_layout()
plt.show()
# Private rooms have the higher median (12 against 8). The boxes are low (they end at 27 and 38)
# and the whiskers are short compared with all the dots above them.

## Task 5: The bar charts

In [ ]:
# (a) number of listings per room type, biggest first
order = listings['room_type'].value_counts().index
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=listings, y='room_type', order=order, ax=ax)
ax.set_title('Almost all listings are entire homes or private rooms')
ax.set_xlabel('Number of listings')
ax.set_ylabel('Room type')
plt.tight_layout()
plt.show()
# Entire home/apt 3760, Private room 787, Shared room 24, Hotel room 3

In [ ]:
# (b) median reviews and number of listings per neighbourhood
table = (listings.groupby('neighbourhood_cleansed')['number_of_reviews']
                 .agg(median='median', n='count')
                 .sort_values('median', ascending=False)
                 .reset_index())
table['label'] = table['neighbourhood_cleansed'] + ' (' + table['n'].astype(str) + ')'
print(table.head(4))

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=table, x='median', y='label', ax=ax)
ax.set_title('Södermalms has the highest median number of reviews')
ax.set_xlabel('Median number of reviews')
ax.set_ylabel('Neighbourhood (number of listings)')
plt.tight_layout()
plt.show()
# Highest median: Södermalms 17 (1145 listings), then Kungsholmens 12, Norrmalms 10, Östermalms 9.
# Lowest: Skärholmens 3 (96 listings). Södermalms has both the highest median and the most listings.

## Task 6: The scatter plot

In [ ]:
print('Outside the picture (more than 300 reviews):', (reviews > 300).sum())     # 91

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=listings, x='availability_365', y='number_of_reviews', alpha=0.3, ax=ax)
ax.set_ylim(0, 300)
ax.set_title('Availability and reviews: a weak connection (91 listings above 300 are outside)')
ax.set_xlabel('Days available in the next 365 days')
ax.set_ylabel('Number of reviews')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=big, x='availability_365', y='number_of_reviews', hue='room_type', alpha=0.3, ax=ax)
ax.set_ylim(0, 300)
ax.set_title('Entire homes and private rooms overlap: no clear difference in the pattern')
ax.set_xlabel('Days available in the next 365 days')
ax.set_ylabel('Number of reviews')
plt.tight_layout()
plt.show()
# I see a cloud of dots, densest at the bottom (few reviews), spread over the whole x-axis from 0 to 365.
# There is no clear line or slope, so the pattern is weak. That fits the correlation of 0.18 from lab 6.
# In the second chart the two room types overlap a lot.

## Task 7: The heatmap and the line chart

In [ ]:
cols = ['number_of_reviews', 'price', 'accommodates', 'bedrooms', 'availability_365',
        'review_scores_rating', 'minimum_nights', 'days_since_review']
corr = listings[cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
ax.set_title('Number of reviews has only weak connections; the strong ones are size and price')
plt.tight_layout()
plt.show()
# The first row (number_of_reviews) has only weak numbers: the strongest are days_since_review (-0.19)
# and availability_365 (0.18). The strong squares are elsewhere: accommodates and bedrooms 0.78,
# and price with both of them 0.58.

In [ ]:
per_year = listings['last_review'].dt.year.value_counts().sort_index()
print(per_year.tail(4))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(x=per_year.index, y=per_year.values, marker='o', ax=ax)
ax.set_title('Most listings were reviewed last in 2026 (the year is not complete)')
ax.set_xlabel('Year of the last review')
ax.set_ylabel('Number of listings')
plt.tight_layout()
plt.show()
# The line is nearly flat from 2014 to 2022 (a few dozen listings per year), then rises, and 2026 is far
# higher: 2705 listings had their last review in 2026. That means these listings are still active and
# still reviewed. And the year is not even complete: the data ends in September.

## Task 8: A chart that lies, and a chart that does not

In [ ]:
two = listings[listings['room_type'].isin(['Entire home/apt', 'Private room']) & listings['has_reviews']]
med = two.groupby('room_type')['review_scores_rating'].median()
cnt = two.groupby('room_type')['review_scores_rating'].count()
print(med.round(2).to_dict(), cnt.to_dict())     # Entire 4.92 (3229), Private 4.89 (675)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x=med.index, y=med.values, ax=axes[0])
axes[0].set_ylim(0, 5)
axes[0].set_title('Honest: axis from 0, the two bars look the same')
sns.barplot(x=med.index, y=med.values, ax=axes[1])
axes[1].set_ylim(4.88, 4.93)
axes[1].set_title('Misleading: axis cut at 4.88, a tiny gap looks huge')
for ax in axes:
    ax.set_xlabel('Room type')
    ax.set_ylabel('Median rating')
plt.tight_layout()
plt.show()
# The second chart makes me believe entire homes are rated MUCH better: the bar looks about four times as
# high as the private room bar (0.04 against 0.01). The real difference is only 0.03 on a scale from 1 to 5.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.barplot(x=med.index, y=med.values, ax=ax)
ax.set_ylim(0, 5)
ax.set_title(f'Median rating is almost the same for both room types\n(3229 and 675 rated listings)')
ax.set_xlabel('Room type')
ax.set_ylabel('Median rating (scale 1 to 5)')
plt.tight_layout()
fig.savefig('rating_per_room_type.png', dpi=150, bbox_inches='tight')
plt.show()
print(os.path.exists('rating_per_room_type.png'))     # True: the file is in my folder

# Part 2: Go deeper

## Task 1: A shape that tells a story

In [ ]:
rated = listings[listings['has_reviews']]        # only the listings with a rating
print('Outside the picture (more than 300 reviews):', (rated['number_of_reviews'] > 300).sum())   # 91
many = rated[rated['number_of_reviews'] >= 100]
few = rated[rated['number_of_reviews'] <= 3]
print(len(many), many['review_scores_rating'].min())     # 431 listings, lowest rating 4.18
print(len(few), few['review_scores_rating'].min())       # 798 listings, lowest rating 1.0

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.scatterplot(data=rated, x='number_of_reviews', y='review_scores_rating', alpha=0.3, ax=ax)
ax.set_xlim(0, 300)
ax.set_ylim(0.9, 5.1)
ax.set_title('Few reviews, any rating: the more reviews, the more stable the rating')
ax.set_xlabel('Number of reviews')
ax.set_ylabel('Rating (1 to 5)')
plt.tight_layout()
plt.show()
# The cloud is a funnel. On the left (few reviews) the dots are spread from 1 to 5, and many lie at exactly 5.
# The more reviews, the more the dots gather between about 4.3 and 5. With 100 reviews or more the lowest
# rating is 4.18, with 3 reviews or fewer it is 1.0. 91 listings (more than 300 reviews) are outside the picture.
# A listing with few reviews can have any rating because one guest decides. With many reviews the rating is stable.

## Task 2: Do superhosts have more reviews?

In [ ]:
g = listings.groupby('host_is_superhost')['number_of_reviews']
print(g.count().to_dict(), g.median().to_dict())      # others 3837, median 7. superhosts 737, median 53
out = listings[listings['number_of_reviews'] > 300].groupby('host_is_superhost').size()
print(out.to_dict())                                  # others 44, superhosts 47 (91 in total)

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=listings, x='host_is_superhost', y='number_of_reviews', ax=ax)
ax.set_ylim(0, 300)
ax.set_xticks([0, 1])
ax.set_xticklabels([f'Other hosts\n(3837 listings)', f'Superhosts\n(737 listings)'])
ax.set_title('Superhosts have a much higher median number of reviews (53 against 7)\n91 listings above 300 are outside the picture')
ax.set_xlabel('')
ax.set_ylabel('Number of reviews')
plt.tight_layout()
plt.show()
# I CAN say: superhosts have many more reviews. The box of the superhosts is much higher.
# I CANNOT say: that being a superhost gives more reviews. The badge may depend on the reviews
# (as far as I know, it needs enough bookings), and the chart does not show why.

## Task 3: Fix a chart that lies

**(a) What is wrong with the colleague's chart**
1. The y-axis is cut at 4.89, so a tiny difference looks enormous.
2. The title says "Superhosts are rated lower", but the real difference is only 0.02 on a scale from 1 to 5.
3. There are no names on the axes, so I do not know what the numbers mean.
4. The number of listings behind each bar is not shown.
5. It uses the median without saying so, and one number per group hides the spread.

In [ ]:
# (c) median, mean and number of listings for each group
stats = listings.groupby('host_is_superhost')['review_scores_rating'].agg(['median', 'mean', 'count']).round(2)
stats
# Other hosts: median 4.92, mean 4.81, 3215 rated listings. Superhosts: median 4.90, mean 4.87, 715 rated listings.
# Strange: the direction changes. By the median the superhosts are lower, by the mean they are higher.
# So the difference is not a stable result at all.

In [ ]:
# (b) the honest chart: axis from 0, a neutral title, axis names, counts. A boxplot next to it shows the spread.
medians = listings.groupby('host_is_superhost')['review_scores_rating'].median()
labels = ['Other hosts\n(3215 rated)', 'Superhosts\n(715 rated)']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x=labels, y=medians.values, ax=axes[0])
axes[0].set_ylim(0, 5)
axes[0].set_title('Superhosts and other hosts have almost the same median rating')
axes[0].set_ylabel('Median rating (scale 1 to 5)')
sns.boxplot(data=listings[listings['has_reviews']], x='host_is_superhost', y='review_scores_rating', ax=axes[1])
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(labels)
axes[1].set_title('Other hosts have a wider spread, with some very low ratings')
axes[1].set_ylabel('Rating')
for ax in axes:
    ax.set_xlabel('')
plt.tight_layout()
fig.savefig('superhost_rating.png', dpi=150, bbox_inches='tight')
plt.show()
# A bar chart of one number per group hides the spread. The boxplot shows what the bars hide: other hosts have
# some very low ratings (down to 1.0), while the superhosts are closer together near 4.9 (lowest about 3.5).

## Task 4: Your own chart

In [ ]:
# Question: In which years did the listings that are in the table get their first review?
first_year = listings['first_review'].dt.year.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(x=first_year.index, y=first_year.values, marker='o', ax=ax)
ax.set_title('Most of today\'s listings got their first review in 2022 or later')
ax.set_xlabel('Year of the first review')
ax.set_ylabel('Number of listings')
plt.tight_layout()
plt.show()
# Why this chart: a line chart is good for counts over time.
# What I left out: the 644 listings without any review (they have no date), and 2026 is not a complete year.
# What it does NOT tell: how many listings were active in each year. It only shows listings that still exist
# in the table today. Listings that closed years ago are missing, so the old years look smaller than they were.
# (Replace this with a question of your own!)